# 01 — Classic RAG with local models

**Learning goal:** see the complete retrieve-then-generate pipeline: configure a profile, ingest its documents into in-memory Chroma, inspect nearest passages, build a grounded prompt, and ask a local chat model.

**Prerequisites:** Python 3, this notebook inside the `demos/` folder, and LM Studio with the configured chat and embedding models loaded. Run cells top to bottom. Set `PROFILE` to `"onia"` or `"devtalks"` before ingestion. The install, configuration, corpus, and prompt-builder cells work offline; cells marked **LM Studio** are opt-in and require the local server. `RUN_LM_STUDIO_DEMO` is intentionally `False` so a safe Run All performs no model calls.

In [ ]:
%pip install -q openai==2.53.0 chromadb==1.5.9

## 1. Configuration
Edit the constants here or override them with environment variables of the same name. Document discovery starts at `Path.cwd()` and accepts either the `demos/` directory, its parent, or a descendant.

In [ ]:
import os
from pathlib import Path

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://127.0.0.1:1234/v1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "lm-studio")
CHAT_MODEL = os.getenv("CHAT_MODEL", "qwen/qwen3.5-9b")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-qwen3-embedding-4b")
PROFILE = os.getenv("PROFILE", "devtalks").lower()  # "onia" or "devtalks"
TOP_K = int(os.getenv("TOP_K", "3"))
RUN_LM_STUDIO_DEMO = False

if PROFILE not in {"onia", "devtalks"}:
    raise ValueError("PROFILE must be 'onia' or 'devtalks'.")
if TOP_K <= 0:
    raise ValueError("TOP_K must be a positive integer.")

def find_demo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "demos", *cwd.parents]
    for candidate in candidates:
        if (candidate / "documents" / "shared" / "rag_overview.md").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find demos/documents from the current working directory. "
        "Launch Jupyter from the presentations root or the demos directory."
    )

DEMO_ROOT = find_demo_root()
DOCUMENTS_ROOT = DEMO_ROOT / "documents"
print(f"Profile: {PROFILE} | documents: {DOCUMENTS_ROOT}")

## 2. Select the profile-aware corpus
ONIA uses its classic-RAG model note; DevTalks uses the shared model note. Each profile combines shared RAG/LM Studio material with its own conference document.

In [ ]:
PROFILE_DOCUMENTS = {
    "onia": [
        "shared/lm_studio.md",
        "shared/rag_overview.md",
        "onia/qwen_models_classic_rag.md",
        "onia/onia_conference.md",
    ],
    "devtalks": [
        "shared/lm_studio.md",
        "shared/rag_overview.md",
        "shared/qwen_models.md",
        "devtalks/devtalks_conference.md",
    ],
}

document_paths = [DOCUMENTS_ROOT / relative for relative in PROFILE_DOCUMENTS[PROFILE]]
missing = [path for path in document_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing demo documents: {missing}")
documents = [(path.relative_to(DOCUMENTS_ROOT).as_posix(), path.read_text(encoding="utf-8")) for path in document_paths]
print("Corpus:")
for source, text in documents:
    print(f"  {source}: {len(text)} characters")

## 3. Connection and model check — **requires LM Studio**
Enable the guard to contact the local OpenAI-compatible endpoint and verify that both configured models are advertised.

In [ ]:
from openai import OpenAI

openai_client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)

if RUN_LM_STUDIO_DEMO:
    available_models = sorted(model.id for model in openai_client.models.list().data)
    print("Available models:", available_models)
    for required_model in (CHAT_MODEL, EMBEDDING_MODEL):
        if required_model not in available_models:
            raise RuntimeError(f"Load {required_model!r} in LM Studio before continuing.")
else:
    print("Skipped. Set RUN_LM_STUDIO_DEMO=True to check the local server.")

## 4. Ingest into in-memory Chroma — **requires LM Studio embeddings**
Chroma calls the embedding adapter for both stored documents and later queries, keeping vectors in the same space. The collection disappears with the kernel.

In [ ]:
import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings

class OpenAIEmbeddingFunction(EmbeddingFunction[Documents]):
    def __init__(self, client: OpenAI, model: str) -> None:
        self.client = client
        self.model = model

    def __call__(self, input: Documents) -> Embeddings:
        response = self.client.embeddings.create(model=self.model, input=list(input))
        return [item.embedding for item in response.data]

collection = None
if RUN_LM_STUDIO_DEMO:
    chroma_client = chromadb.EphemeralClient()
    embedder = OpenAIEmbeddingFunction(openai_client, EMBEDDING_MODEL)
    collection = chroma_client.get_or_create_collection(
        name=f"notebook_rag_{PROFILE}", embedding_function=embedder
    )
    collection.add(
        ids=[f"doc-{index}" for index in range(len(documents))],
        documents=[text for _, text in documents],
        metadatas=[{"source": source} for source, _ in documents],
    )
    print(f"Indexed {collection.count()} documents.")
else:
    print("Skipped ingestion.")

## 5. Retrieve and inspect — **requires LM Studio embeddings**
Change the question for the selected profile. Distances are shown so the retrieval step stays visible rather than becoming hidden prompt plumbing.

In [ ]:
QUESTION = (
    "What is ONIA and which models support this RAG demo?"
    if PROFILE == "onia"
    else "What is DevTalks and which models support this RAG demo?"
)
hits = []
if RUN_LM_STUDIO_DEMO:
    result = collection.query(query_texts=[QUESTION], n_results=min(TOP_K, collection.count()))
    hits = [
        {"text": text, "source": metadata["source"], "distance": distance}
        for text, metadata, distance in zip(
            result["documents"][0], result["metadatas"][0], result["distances"][0]
        )
    ]
    for rank, hit in enumerate(hits, 1):
        preview = hit["text"].replace("\n", " ")[:120]
        print(f"{rank}. {hit['source']} | distance={hit['distance']:.4f} | {preview}…")
else:
    print("Skipped retrieval.")

## 6. Build the grounded prompt
The instruction limits generation to retrieved evidence and asks for filename citations. This function itself is offline; the preview appears after retrieval.

In [ ]:
def grounded_messages(question: str, retrieved_hits: list[dict]) -> list[dict]:
    context = "\n\n".join(
        f"[source: {hit['source']}]\n{hit['text']}" for hit in retrieved_hits
    )
    return [
        {
            "role": "system",
            "content": (
                "Answer only from the supplied context. If it does not contain the answer, "
                "say you do not know. Cite supporting source filenames."
            ),
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

messages = grounded_messages(QUESTION, hits) if hits else []
print(messages[1]["content"][:800] if messages else "Prompt awaits retrieved passages.")

## 7. Generate the answer — **requires LM Studio chat**
The final call uses the retrieved context assembled above.

In [ ]:
if RUN_LM_STUDIO_DEMO:
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL, messages=messages, temperature=0.2
    )
    print(response.choices[0].message.content or "")
else:
    print("Skipped generation. Set RUN_LM_STUDIO_DEMO=True and rerun cells 3–7.")